In [4]:
import time
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.preprocessing import StandardScaler, OneHotEncoder, MinMaxScaler

np.random.seed(0)

In [5]:
def relu(x): return np.maximum(0, x)
def drelu(x): return (x > 0).astype(float)
def sigmoid(x): return 1.0 / (1.0 + np.exp(-x))
def softmax(z):
    z = z - np.max(z, axis=1, keepdims=True)
    e = np.exp(z)
    return e / e.sum(axis=1, keepdims=True)

In [6]:
def mse_loss(yhat, y): 
    return np.mean((yhat - y)**2)

def bce_loss(yhat, y, eps=1e-9):
    yhat = np.clip(yhat, eps, 1-eps)
    return -np.mean(y * np.log(yhat) + (1-y) * np.log(1-yhat))

def cce_loss(p, y_onehot, eps=1e-9):
    p = np.clip(p, eps, 1.0)
    return -np.mean(np.sum(y_onehot * np.log(p), axis=1))

def init_params(din, dh, dout, scale=0.1):
    W1 = np.random.randn(din, dh) * scale
    b1 = np.zeros((1, dh))
    W2 = np.random.randn(dh, dout) * scale
    b2 = np.zeros((1, dout))
    return W1, b1, W2, b2

In [7]:
def train_regression_boston(epochs=200, lr=0.01):
    boston = fetch_openml(name="Boston", version=1, as_frame=False)
    X = StandardScaler().fit_transform(boston["data"])
    y = MinMaxScaler((0,1)).fit_transform(boston["target"].reshape(-1,1))
    W1,b1,W2,b2 = init_params(X.shape[1], 64, 1)
    t0 = time.perf_counter()
    for ep in range(1, epochs+1):
        h = relu(X @ W1 + b1)
        yhat = sigmoid(h @ W2 + b2)
        loss = mse_loss(yhat, y)
        # backprop
        N = X.shape[0]
        dloss = 2*(yhat - y)/N
        dz = dloss * (yhat * (1 - yhat))
        dW2 = h.T @ dz
        db2 = np.sum(dz, axis=0, keepdims=True)
        dh = dz @ W2.T
        dpre = dh * drelu(h)
        dW1 = X.T @ dpre
        db1 = np.sum(dpre, axis=0, keepdims=True)
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1
        if ep % 10 == 0 or ep == 1 or ep == epochs:
            print(f"[Boston][ep {ep}/{epochs}] loss={loss:.6f} elapsed={time.perf_counter()-t0:0.1f}s")
    print("Boston done.\n")

In [8]:
def train_mnist_multiclass(epochs=5, lr=0.1):
    mn = fetch_openml("mnist_784", version=1, as_frame=False)
    X = mn["data"].astype(np.float32) / 255.0
    y = mn["target"].astype(int)
    X = StandardScaler().fit_transform(X)
    enc = OneHotEncoder(sparse_output=False)
    yoh = enc.fit_transform(y.reshape(-1,1))
    W1,b1,W2,b2 = init_params(X.shape[1], 128, yoh.shape[1], scale=0.05)
    N = X.shape[0]
    t0 = time.perf_counter()
    for ep in range(1, epochs+1):
        h = relu(X @ W1 + b1)
        logits = h @ W2 + b2
        p = softmax(logits)
        loss = cce_loss(p, yoh)
        dz = (p - yoh) / N
        dW2 = h.T @ dz
        db2 = np.sum(dz, axis=0, keepdims=True)
        dh = dz @ W2.T
        dpre = dh * drelu(h)
        dW1 = X.T @ dpre
        db1 = np.sum(dpre, axis=0, keepdims=True)
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1
        if ep % 1 == 0:
            preds = np.argmax(p, axis=1)
            acc = np.mean(preds == y)
            print(f"[MNIST][ep {ep}/{epochs}] loss={loss:.4f} acc={acc:.4f} elapsed={time.perf_counter()-t0:0.1f}s")
    print("MNIST done.\n")

In [9]:
def train_pima_binary(epochs=300, lr=0.01):
    data = fetch_openml("pima-indians-diabetes", version=1, as_frame=False)
    X = StandardScaler().fit_transform(data["data"])
    y = data["target"].astype(float).reshape(-1,1)
    W1,b1,W2,b2 = init_params(X.shape[1], 32, 1)
    t0 = time.perf_counter()
    N = X.shape[0]
    for ep in range(1, epochs+1):
        h = relu(X @ W1 + b1)
        yhat = sigmoid(h @ W2 + b2)
        loss = bce_loss(yhat, y)
        dz = (yhat - y) / N
        dW2 = h.T @ dz
        db2 = np.sum(dz, axis=0, keepdims=True)
        dh = dz @ W2.T
        dpre = dh * drelu(h)
        dW1 = X.T @ dpre
        db1 = np.sum(dpre, axis=0, keepdims=True)
        W2 -= lr * dW2; b2 -= lr * db2
        W1 -= lr * dW1; b1 -= lr * db1
        if ep % 50 == 0 or ep == 1 or ep == epochs:
            preds = (yhat > 0.5).astype(int)
            acc = np.mean(preds == y)
            print(f"[Pima][ep {ep}/{epochs}] loss={loss:.6f} acc={acc:.4f} elapsed={time.perf_counter()-t0:0.1f}s")
    print("Pima done.\n")

In [10]:
if __name__ == "__main__":
    train_regression_boston()
    train_mnist_multiclass()
    train_pima_binary()

[Boston][ep 1/200] loss=0.050660 elapsed=0.0s
[Boston][ep 10/200] loss=0.049687 elapsed=0.0s
[Boston][ep 20/200] loss=0.048648 elapsed=0.0s
[Boston][ep 30/200] loss=0.047647 elapsed=0.0s
[Boston][ep 40/200] loss=0.046686 elapsed=0.0s
[Boston][ep 50/200] loss=0.045762 elapsed=0.0s
[Boston][ep 60/200] loss=0.044873 elapsed=0.0s
[Boston][ep 70/200] loss=0.044018 elapsed=0.0s
[Boston][ep 80/200] loss=0.043195 elapsed=0.0s
[Boston][ep 90/200] loss=0.042403 elapsed=0.0s
[Boston][ep 100/200] loss=0.041639 elapsed=0.0s
[Boston][ep 110/200] loss=0.040904 elapsed=0.0s
[Boston][ep 120/200] loss=0.040199 elapsed=0.0s
[Boston][ep 130/200] loss=0.039520 elapsed=0.0s
[Boston][ep 140/200] loss=0.038867 elapsed=0.0s
[Boston][ep 150/200] loss=0.038237 elapsed=0.0s
[Boston][ep 160/200] loss=0.037630 elapsed=0.0s
[Boston][ep 170/200] loss=0.037046 elapsed=0.1s
[Boston][ep 180/200] loss=0.036481 elapsed=0.1s
[Boston][ep 190/200] loss=0.035936 elapsed=0.1s
[Boston][ep 200/200] loss=0.035408 elapsed=0.1s
Bos